<a href="https://colab.research.google.com/github/sahanyafernando/EN3150-A03-edge-cnn/blob/Rajitha/notebooks/01_model_a_standard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Model A Standard CNN

**EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification**  
**Owner:** Rajitha  
**Environment:** Google Colab + TensorFlow/Keras

## Goal

Build and train **Model A — the Standard CNN baseline** for the 5-class TF Flowers classification task.

This notebook:
- loads the same shared dataset configuration used by the group,
- builds a standard `Conv2D + MaxPooling2D` CNN,
- trains it for **at least 20 epochs**,
- plots training/validation curves,
- evaluates test accuracy, macro precision, macro recall, and confusion matrix,
- records parameter count, approximate MACs, model size, epoch time, and inference time,
- exports `model_a.json` to the group's shared Google Drive results folder.

Model A is the **baseline** that will later be compared with the lightweight depthwise-separable Model B.

In [1]:
%pip install -q scikit-learn

In [2]:
import os
import gc
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ----------------------------------------------------
# GROUP-SHARED SETTINGS — do not change independently
# ----------------------------------------------------
SEED = 42
IMG_SIZE = 64
BATCH_SIZE = 64

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("SEED =", SEED)
print("IMG_SIZE =", IMG_SIZE)
print("BATCH_SIZE =", BATCH_SIZE)

SEED = 42
IMG_SIZE = 64
BATCH_SIZE = 64


In [4]:
from google.colab import drive
drive.mount("/content/drive")

# ----------------------------------------------------
# Rajitha's personal storage
# ----------------------------------------------------
MEMBER = "rajitha"

PERSONAL_ROOT = Path(
    f"/content/drive/MyDrive/EN3150_A03_PERSONAL/{MEMBER}"
)

DATA_ROOT = PERSONAL_ROOT / "dataset_cache"
ARTIFACT_ROOT = PERSONAL_ROOT / "artifacts"
PLOT_ROOT = PERSONAL_ROOT / "plots"

for p in [DATA_ROOT, ARTIFACT_ROOT, PLOT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# Group-shared result storage
# ----------------------------------------------------
# Before running this notebook, Rajitha should add the group's
# EN3150_A03_SHARED folder as a shortcut inside My Drive.
SHARED_ROOT = Path(
    "/content/drive/MyDrive/EN3150_A03_SHARED"
)

if not SHARED_ROOT.exists():
    raise FileNotFoundError(
        "Shared folder not found. Add the group's 'EN3150_A03_SHARED' "
        "folder as a shortcut inside My Drive, then rerun this cell."
    )

RESULT_ROOT = SHARED_ROOT / "shared_results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("Personal root:", PERSONAL_ROOT)
print("Shared result root:", RESULT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Personal root: /content/drive/MyDrive/EN3150_A03_PERSONAL/rajitha
Shared result root: /content/drive/MyDrive/EN3150_A03_SHARED/shared_results


In [5]:
# ============================================================
# TF FLOWERS DATASET — same robust loader used by the group
# ============================================================

DATASET_URL = (
    "https://storage.googleapis.com/download.tensorflow.org/"
    "example_images/flower_photos.tgz"
)

EXPECTED_CLASSES = {
    "daisy",
    "dandelion",
    "roses",
    "sunflowers",
    "tulips",
}

# Download once and cache in Rajitha's personal Google Drive.
archive_path = keras.utils.get_file(
    fname="flower_photos.tgz",
    origin=DATASET_URL,
    extract=True,
    cache_dir=str(DATA_ROOT),
    cache_subdir="downloads",
)

archive_path = Path(archive_path)
search_root = archive_path.parent

# ------------------------------------------------------------
# Find the real folder that contains the five class folders.
# This handles different Keras extraction layouts.
# ------------------------------------------------------------
DATA_DIR = None

candidate_dirs = [search_root]
candidate_dirs.extend(
    p for p in search_root.rglob("*")
    if p.is_dir()
)

for candidate in candidate_dirs:
    try:
        child_dirs = {
            p.name
            for p in candidate.iterdir()
            if p.is_dir()
        }
    except PermissionError:
        continue

    if EXPECTED_CLASSES.issubset(child_dirs):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find the extracted TF Flowers class folders. "
        f"Searched inside: {search_root}"
    )

print("Dataset directory:", DATA_DIR)

# ------------------------------------------------------------
# Collect image paths and labels in a deterministic order.
# ------------------------------------------------------------
CLASS_NAMES = sorted(EXPECTED_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)

class_to_index = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

all_paths = []
all_labels = []

valid_extensions = {".jpg", ".jpeg", ".png"}

for class_name in CLASS_NAMES:
    class_dir = DATA_DIR / class_name

    for image_path in sorted(class_dir.rglob("*")):
        if (
            image_path.is_file()
            and image_path.suffix.lower() in valid_extensions
        ):
            all_paths.append(str(image_path))
            all_labels.append(class_to_index[class_name])

# Force filename array to string dtype.
all_paths = np.asarray(all_paths, dtype=str)
all_labels = np.asarray(all_labels, dtype=np.int32)

print("Classes:", CLASS_NAMES)
print("Total images:", len(all_paths))

assert len(all_paths) > 0, "No flower images were found."
assert NUM_CLASSES == 5

# ------------------------------------------------------------
# SAME deterministic 70% / 15% / 15% split as the group notebook.
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)

indices = np.arange(len(all_paths))
rng.shuffle(indices)

all_paths = all_paths[indices]
all_labels = all_labels[indices]

n_total = len(all_paths)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_paths = all_paths[:n_train]
train_labels = all_labels[:n_train]

val_paths = all_paths[n_train:n_train + n_val]
val_labels = all_labels[n_train:n_train + n_val]

test_paths = all_paths[n_train + n_val:]
test_labels = all_labels[n_train + n_val:]

def decode_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
        try_recover_truncated=True,
    )
    return image, label

raw_train = (
    tf.data.Dataset
    .from_tensor_slices((train_paths, train_labels))
    .map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
)

raw_val = (
    tf.data.Dataset
    .from_tensor_slices((val_paths, val_labels))
    .map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
)

raw_test = (
    tf.data.Dataset
    .from_tensor_slices((test_paths, test_labels))
    .map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
)

def count_examples(ds):
    return int(
        tf.data.experimental.cardinality(ds).numpy()
    )

print("Train:", count_examples(raw_train))
print("Validation:", count_examples(raw_val))
print("Test:", count_examples(raw_test))

assert (
    count_examples(raw_train)
    + count_examples(raw_val)
    + count_examples(raw_test)
    == n_total
)

print("Dataset loaded successfully.")

Dataset directory: /content/drive/MyDrive/EN3150_A03_PERSONAL/rajitha/dataset_cache/downloads/flower_photos_extracted/flower_photos
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
Total images: 3670
Train: 2569
Validation: 550
Test: 551
Dataset loaded successfully.


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        antialias=True,
    )
    image = tf.cast(image, tf.float32)
    return image, label

train_ds = (
    raw_train
    .shuffle(
        2048,
        seed=SEED,
        reshuffle_each_iteration=True,
    )
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    raw_test
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Prepared 64×64 train/validation/test pipelines.")

Prepared 64×64 train/validation/test pipelines.


---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
chore: verify shared dataset setup for model A
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Model A architecture — Standard CNN baseline

Architecture:

```text
64×64×3 input
   ↓
Conv2D(32, 3×3, ReLU)
   ↓
MaxPooling2D
   ↓
Conv2D(64, 3×3, ReLU)
   ↓
MaxPooling2D
   ↓
Conv2D(128, 3×3, ReLU)
   ↓
MaxPooling2D
   ↓
GlobalAveragePooling2D
   ↓
Dense(64, ReLU)
   ↓
Dense(5 logits)
```

This is the **standard-convolution baseline**. Later the group compares it with Model B, where standard convolutions are replaced with depthwise-separable convolutions to reduce parameters and MAC operations.

ReLU is used in hidden layers because it is computationally simple and avoids expensive exponential operations.

In [ ]:
def build_model_a():

    inputs = keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3),
        name="image"
    )

    # Convert pixel values from 0..255 to 0..1.
    x = layers.Rescaling(
        1.0 / 255.0,
        name="rescale"
    )(inputs)

    x = layers.Conv2D(
        32,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="conv1"
    )(x)

    x = layers.MaxPooling2D(
        name="pool1"
    )(x)

    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="conv2"
    )(x)

    x = layers.MaxPooling2D(
        name="pool2"
    )(x)

    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="conv3"
    )(x)

    x = layers.MaxPooling2D(
        name="pool3"
    )(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pool"
    )(x)

    x = layers.Dense(
        64,
        activation="relu",
        name="dense64"
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        name="logits"
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="Model_A_Standard_CNN"
    )


model_a = build_model_a()

model_a.summary()

print("Total parameters:", model_a.count_params())

In [ ]:
def parameter_table(model):
    rows=[]
    for l in model.layers:
        p=int(sum(np.prod(v.shape) for v in l.trainable_weights))
        if p: rows.append({'layer':l.name,'type':l.__class__.__name__,'output_shape':str(l.output.shape),'trainable_params':p})
    return pd.DataFrame(rows)
display(parameter_table(model_a))

In [ ]:
def estimate_macs_a(model):
    total=0
    for l in model.layers:
        if isinstance(l,layers.Conv2D):
            h,w,cout=map(int,l.output.shape[1:]); cin=int(l.input.shape[-1]); kh,kw=l.kernel_size; total+=h*w*cout*kh*kw*cin
        elif isinstance(l,layers.Dense): total+=int(l.input.shape[-1])*int(l.units)
    return int(total)
MODEL_A_MACS=estimate_macs_a(model_a); print('Approx MACs:',f'{MODEL_A_MACS:,}')

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
feat: implement standard CNN model A architecture
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Train 20 epochs with resume protection

In [ ]:
LOSS_FN=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
class PersistentEpochTimer(keras.callbacks.Callback):
    def __init__(self,csv_path): super().__init__(); self.csv_path=Path(csv_path); self.csv_path.parent.mkdir(parents=True,exist_ok=True)
    def on_epoch_begin(self,epoch,logs=None): self.start=time.perf_counter()
    def on_epoch_end(self,epoch,logs=None):
        row=pd.DataFrame([{'epoch':int(epoch),'seconds':float(time.perf_counter()-self.start)}])
        row.to_csv(self.csv_path,mode='a',header=not self.csv_path.exists(),index=False)
def read_log(run_name):
    p=ARTIFACT_ROOT/run_name/'training_log.csv'
    if not p.exists(): return pd.DataFrame()
    d=pd.read_csv(p)
    return d.drop_duplicates(subset=['epoch'],keep='last').sort_values('epoch') if 'epoch' in d else d
def average_epoch_time(run_name):
    p=ARTIFACT_ROOT/run_name/'epoch_times.csv'
    if not p.exists(): return np.nan
    d=pd.read_csv(p).drop_duplicates(subset=['epoch'],keep='last')
    return float(d.seconds.mean()) if len(d) else np.nan
def fit_resumable(model,run_name,optimizer,epochs):
    rd=ARTIFACT_ROOT/run_name; rd.mkdir(parents=True,exist_ok=True)
    final=rd/'final.keras'; best=rd/'best.keras'; backup=rd/'backup'; log=rd/'training_log.csv'; timing=rd/'epoch_times.csv'
    if final.exists(): print('Completed run found:',run_name); return keras.models.load_model(final)
    model.compile(optimizer=optimizer,loss=LOSS_FN,metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    callbacks=[keras.callbacks.BackupAndRestore(backup_dir=str(backup),save_freq='epoch',delete_checkpoint=False),keras.callbacks.ModelCheckpoint(str(best),monitor='val_accuracy',mode='max',save_best_only=True,verbose=1),keras.callbacks.CSVLogger(str(log),append=True),PersistentEpochTimer(timing)]
    model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,verbose=1)
    model.save(final); return model
def reset_run(run_name):
    import shutil
    p=ARTIFACT_ROOT/run_name
    if p.exists(): shutil.rmtree(p)
def plot_history(run_name,prefix):
    d=read_log(run_name)
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.loss,label='Train'); plt.plot(d.epoch+1,d.val_loss,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(prefix+' Loss'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_loss.png'),dpi=180); plt.show()
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.accuracy,label='Train'); plt.plot(d.epoch+1,d.val_accuracy,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title(prefix+' Accuracy'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_accuracy.png'),dpi=180); plt.show()

In [ ]:
MODEL_A_EPOCHS = 20

model_a = fit_resumable(
    build_model_a(),
    run_name="model_a_adam",
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    epochs=MODEL_A_EPOCHS,
)

plot_history(
    "model_a_adam",
    "Model A"
)

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
exp: train model A for 20 epochs and add learning curves
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Test evaluation

In [ ]:
def get_true_labels(ds): return np.concatenate([y.numpy() for _,y in ds])
Y_TEST=get_true_labels(test_ds)
def model_file_size_mb(path): return Path(path).stat().st_size/(1024**2)
def benchmark_inference_ms_per_image(model,ds,max_batches=10):
    batches=[]; n=0
    for i,(x,_) in enumerate(ds):
        if i>=max_batches: break
        batches.append(x); n+=int(x.shape[0])
    if not batches: return np.nan
    _=model(batches[0],training=False); t=time.perf_counter()
    for x in batches: _=model(x,training=False)
    return (time.perf_counter()-t)*1000/n
def evaluate_model(model,name,save_name):
    pred=np.argmax(model.predict(test_ds,verbose=0),axis=1)
    out={'accuracy':float(accuracy_score(Y_TEST,pred)),'precision_macro':float(precision_score(Y_TEST,pred,average='macro',zero_division=0)),'recall_macro':float(recall_score(Y_TEST,pred,average='macro',zero_division=0))}
    print(out); print(classification_report(Y_TEST,pred,target_names=CLASS_NAMES,digits=4,zero_division=0))
    disp=ConfusionMatrixDisplay(confusion_matrix(Y_TEST,pred),display_labels=CLASS_NAMES); disp.plot(xticks_rotation=45); plt.title(name+' — Confusion Matrix'); plt.tight_layout(); plt.savefig(PLOT_ROOT/save_name,dpi=180); plt.show(); return out

In [ ]:
best_model_path = (
    ARTIFACT_ROOT
    / "model_a_adam"
    / "best.keras"
)

best = keras.models.load_model(
    best_model_path
)

metrics = evaluate_model(
    best,
    "Model A",
    "model_a_confusion_matrix.png"
)

log = read_log(
    "model_a_adam"
)

result = {
    "model": "Model A — Standard CNN",
    "owner": "Rajitha",
    "optimizer": "Adam (lr=0.001)",

    "parameters": int(
        best.count_params()
    ),

    "trainable_parameters": int(
        sum(
            np.prod(v.shape)
            for v in best.trainable_weights
        )
    ),

    "model_size_mb": float(
        model_file_size_mb(
            ARTIFACT_ROOT
            / "model_a_adam"
            / "final.keras"
        )
    ),

    "estimated_fp32_weight_kb": float(
        best.count_params() * 4 / 1024
    ),

    "best_val_accuracy": float(
        log.val_accuracy.max()
    ),

    "accuracy": metrics["accuracy"],

    "precision_macro": metrics[
        "precision_macro"
    ],

    "recall_macro": metrics[
        "recall_macro"
    ],

    "avg_epoch_time_s": float(
        average_epoch_time(
            "model_a_adam"
        )
    ),

    "inference_ms_per_image": float(
        benchmark_inference_ms_per_image(
            best,
            test_ds
        )
    ),

    "approx_macs": int(
        MODEL_A_MACS
    ),
}

result_file = (
    RESULT_ROOT
    / "model_a.json"
)

with open(
    result_file,
    "w"
) as f:
    json.dump(
        result,
        f,
        indent=2
    )

print(
    json.dumps(
        result,
        indent=2
    )
)

print(
    "Shared result saved to:",
    result_file
)

## Interpretation

After the run finishes, Rajitha should write a short discussion based on the **actual measured values**:

1. Did training and validation loss decrease normally?
2. Is there evidence of overfitting or underfitting?
3. What is the final test accuracy?
4. Which flower classes are most often confused in the confusion matrix?
5. What are the macro precision and macro recall?
6. How many parameters does the standard CNN have?
7. What is its model size and approximate MAC count?
8. What is the average training time per epoch?
9. Why is this model useful as the baseline for comparing the lightweight Model B?

Do not write a generic conclusion before the measured results are available.

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
analysis: add model A test metrics confusion matrix and result export
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.